# Phase 3b: CascadeGPS Classifier

From-scratch GraphGPS variant tailored to rumor cascades — see `models/cascade_gps.py`.

Key differences from `GPSClassifier` (notebook 03):
- Edge-aware attention: `edge_attr` (Δt + direction flag) is projected into per-head bias added to Q·Kᵀ.
- Pairwise |Δt| attention bias across the dense batch.
- Per-node gated fusion of MPNN + attention branches.
- Direction-gated GINE-style MPNN (multiplicative edge gate).
- Sinusoidal time encoding on the raw timestamp.
- Learnable ROOT bias added to graph-local node 0.

Runs on Twitter15 and Twitter16 with the same 5 seeds, split, and trainer config as the stock GPS notebook for an apples-to-apples comparison.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from data.dataset import CascadeDataset
from models.cascade_gps import CascadeGPSClassifier
from training.trainer import run_experiment

In [ ]:
DATA_ROOT = "../Twitter15_Twitter16"
SEEDS = [0, 1, 2, 3, 4]

DEVICE0 = "cuda:0" if torch.cuda.is_available() else "cpu"
DEVICE1 = "cuda:1" if torch.cuda.device_count() > 1 else DEVICE0
DEVICE = DEVICE0  # kept for any single-GPU cells
print(f"GPU 0: {DEVICE0}  |  GPU 1: {DEVICE1}")

In [ ]:
ds15 = CascadeDataset(root=DATA_ROOT, name="twitter15")
ds16 = CascadeDataset(root=DATA_ROOT, name="twitter16")
print(f"Twitter15: {len(ds15)} graphs, x={ds15[0].x.shape}, edge_attr={ds15[0].edge_attr.shape}")
print(f"Twitter16: {len(ds16)} graphs, x={ds16[0].x.shape}, edge_attr={ds16[0].edge_attr.shape}")

IN_CHANNELS = ds15[0].x.shape[1]   # 30
EDGE_DIM    = ds15[0].edge_attr.shape[1]  # 2
print(f"in_channels={IN_CHANNELS}, edge_dim={EDGE_DIM}")

In [ ]:
CASCADE_GPS_KWARGS = dict(
    in_channels=IN_CHANNELS,
    hidden_channels=128,
    num_layers=3,
    heads=4,
    dropout=0.1,
    edge_dim=EDGE_DIM,
)

In [ ]:
print("=" * 50)
print("CascadeGPS — Twitter15  (seeds 0–4, full run)")
print("=" * 50)
res15 = run_experiment(
    CascadeGPSClassifier, CASCADE_GPS_KWARGS, ds15, SEEDS,
    epochs=200,
    lr=1e-3,
    weight_decay=0.05,
    patience=60,
    warmup_ratio=0.1,
    lap_pe_sign_flip=True,
    max_nodes_per_batch=2048,
    device=DEVICE,
    out_path="../results/text_off/cascade_gps_twitter15.json",
)
print(f"\nTwitter15 CascadeGPS — Acc: {res15['test_acc_mean']:.3f} ± {res15['test_acc_std']:.3f}  "
      f"F1: {res15['test_f1_mean']:.3f} ± {res15['test_f1_std']:.3f}")


In [ ]:
print("=" * 50)
print("CascadeGPS — Twitter16")
print("=" * 50)
res16 = run_experiment(
    CascadeGPSClassifier, CASCADE_GPS_KWARGS, ds16, SEEDS,
    epochs=200,
    lr=1e-3,
    weight_decay=0.05,
    patience=60,
    warmup_ratio=0.1,
    lap_pe_sign_flip=True,
    max_nodes_per_batch=2048,
    device=DEVICE,
    out_path="../results/text_off/cascade_gps_twitter16.json",
    save_checkpoint=True,
)
print(f"\nTwitter16 CascadeGPS — Acc: {res16['test_acc_mean']:.3f} ± {res16['test_acc_std']:.3f}  "
      f"F1: {res16['test_f1_mean']:.3f} ± {res16['test_f1_std']:.3f}")

In [ ]:
import json, os

def load_result(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

gcn15   = load_result("../results/text_off/gcn_twitter15.json")
gat15   = load_result("../results/text_off/gat_twitter15.json")
bigcn15 = load_result("../results/text_off/bigcn_twitter15.json")
gps15   = load_result("../results/text_off/gps_4way_twitter15.json")
gcn16   = load_result("../results/text_off/gcn_twitter16.json")
gat16   = load_result("../results/text_off/gat_twitter16.json")
bigcn16 = load_result("../results/text_off/bigcn_twitter16.json")
gps16   = load_result("../results/text_off/gps_4way_twitter16.json")

rows = [
    ("GCN",        "Twitter15", gcn15),
    ("GAT",        "Twitter15", gat15),
    ("BiGCN",      "Twitter15", bigcn15),
    ("GPS",        "Twitter15", gps15),
    ("CascadeGPS", "Twitter15", res15),
    ("GCN",        "Twitter16", gcn16),
    ("GAT",        "Twitter16", gat16),
    ("BiGCN",      "Twitter16", bigcn16),
    ("GPS",        "Twitter16", gps16),
    ("CascadeGPS", "Twitter16", res16),
]

print()
print("=" * 70)
print(f"{'Model':<12} {'Dataset':<14} {'Acc mean±std':>16}  {'Macro-F1 mean±std':>18}")
print("-" * 70)
for model, ds_name, res in rows:
    if res is None:
        print(f"{model:<12} {ds_name:<14}  (no results yet)")
        continue
    a_m, a_s = res['test_acc_mean'], res['test_acc_std']
    f_m, f_s = res['test_f1_mean'], res['test_f1_std']
    print(f"{model:<12} {ds_name:<14} {a_m:.3f} ± {a_s:.3f}        {f_m:.3f} ± {f_s:.3f}")
print("=" * 70)
print("(5 seeds, 60/20/20 stratified split, best val-F1 checkpoint)")